In [8]:
import requests
import pandas as pd

def get_coordinates(city_name):
    """Converts a city name like 'Kandy' into Latitude and Longitude."""
    # We append 'Sri Lanka' to ensure it doesn't find a city with the same name in another country!
    url = f"https://nominatim.openstreetmap.org/search?q={city_name}+Sri+Lanka&format=json"
    headers = {'User-Agent': 'UniversityProjectApp/1.0'}

    try:
        response = requests.get(url, headers=headers).json()
        if response:
            lat = float(response[0]['lat'])
            lon = float(response[0]['lon'])
            return lat, lon
        else:
            return None, None
    except Exception as e:
        print(f"Location API Error: {e}")
        return None, None

def get_api_elevation(lat, lon):
    """Uses the exact GPS coordinates to find the height above sea level in meters."""
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    try:
        response = requests.get(url).json()
        return response['elevation'][0] # Returns height in meters
    except Exception as e:
        print(f"Elevation API Error: {e}")
        return 0

In [9]:
def recommend_bike_for_city(city_input, traffic_risk, rainfall_mm):
    print(f"🔍 Searching geography for: {city_input.upper()}...")

    # 1. Get Location
    lat, lon = get_coordinates(city_input)
    if lat is None:
        return f"❌ Could not find the city: '{city_input}'. Please check spelling."

    # 2. Get Elevation
    elevation = get_api_elevation(lat, lon)

    # 3. Format Data for the ML Model (0=Scooter, 1=Geared)
    # Traffic Risk: 0=Low, 1=Medium, 2=High | Rainfall: 0=Dry, 1=Wet
    rain_status = 1 if rainfall_mm > 0 else 0

    input_data = pd.DataFrame([[elevation, traffic_risk, rain_status]],
                              columns=['Elevation', 'Traffic_Risk', 'Rainfall'])

    # 4. Get AI Prediction
    prediction = recommender_model.predict(input_data)[0]

    # 5. Format Output
    print("=" * 45)
    print("🌍 GEOGRAPHY DETECTED:")
    print(f"   City: {city_input.title()}")
    print(f"   Elevation: {elevation} meters above sea level")
    print("-" * 45)

    if prediction == 1:
        print("🎯 AI RECOMMENDS: Geared Motorcycle 🏍️")
        print("   Reason: High elevation detected. A manual gear system provides crucial engine-braking for safe downhill riding.")
    else:
        print("🎯 AI RECOMMENDS: Automatic Scooter 🛵")
        print("   Reason: Flat terrain or traffic conditions detected. Automatic transmission is easier to handle here.")
    print("=" * 45, "\n")

In [10]:
# Test 1: Coastal City (Should detect very low elevation -> Scooter)
# Let's pretend it has Low Traffic (0) and No Rain (0)
recommend_bike_for_city(city_input="Mirissa", traffic_risk=0, rainfall_mm=0)

# Test 2: High Mountain City (Should detect >1000m elevation -> Geared Bike)
# Let's pretend it has Low Traffic (0) and No Rain (0)
recommend_bike_for_city(city_input="Nuwara Eliya", traffic_risk=0, rainfall_mm=0)

# Test 3: Central City (Mid elevation, but let's say traffic is HIGH (2))
recommend_bike_for_city(city_input="Kandy", traffic_risk=2, rainfall_mm=0)

🔍 Searching geography for: MIRISSA...
🌍 GEOGRAPHY DETECTED:
   City: Mirissa
   Elevation: 6.0 meters above sea level
---------------------------------------------
🎯 AI RECOMMENDS: Automatic Scooter 🛵
   Reason: Flat terrain or traffic conditions detected. Automatic transmission is easier to handle here.

🔍 Searching geography for: NUWARA ELIYA...
🌍 GEOGRAPHY DETECTED:
   City: Nuwara Eliya
   Elevation: 2273.0 meters above sea level
---------------------------------------------
🎯 AI RECOMMENDS: Geared Motorcycle 🏍️
   Reason: High elevation detected. A manual gear system provides crucial engine-braking for safe downhill riding.

🔍 Searching geography for: KANDY...
🌍 GEOGRAPHY DETECTED:
   City: Kandy
   Elevation: 505.0 meters above sea level
---------------------------------------------
🎯 AI RECOMMENDS: Geared Motorcycle 🏍️
   Reason: High elevation detected. A manual gear system provides crucial engine-braking for safe downhill riding.



In [11]:
import joblib
from google.colab import files

# Save the model
joblib.dump(recommender_model, 'bike_recommender.joblib')
print("💾 Model saved successfully!")

# Automatically download it to your PC
files.download('bike_recommender.joblib')

💾 Model saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>